# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [4]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import gradio as gr
import sqlite3

In [6]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
claude_api_key = os.getenv('CLAUDE_API_KEY')
Model1 = "claude-sonnet-4-5-20250929"
openai = OpenAI()
anthropic = Anthropic()

OpenAI API Key exists and begins sk-proj-


In [39]:
system_message = """
You are a helpful technical assistant that explains code clearly and concisely.
Always use the look_up_python_docs tool when asked about any specific Python function, 
module, method, or built-in before answering.
Always be accurate. If you don't know the answer, say so.
"""

In [40]:
# Anthropic format (already in Cell 4 - keep as is)
anthropic_tools = [
    {
        "name": "look_up_python_docs",
        "description": "Look up documentation for a Python function or concept",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The Python concept to look up"}
            },
            "required": ["query"]
        }
    }
]
# OpenAI format (add this)
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "look_up_python_docs",
            "description": "Look up documentation for a Python function or concept",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The Python concept to look up"}
                },
                "required": ["query"]
            }
        }
    }
]

In [41]:
def look_up_python_docs(query):
    import pydoc
    result = pydoc.render_doc(query, renderer=pydoc.plaintext)
    return result[:2000] if result else f"No documentation found for '{query}'"

In [48]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=openai_tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        responses = handle_tool_calls_chat(message_obj)
        messages.append(message_obj)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=openai_tools)

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ''
        yield result  

In [49]:
def claude_chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history if h["role"] != "system"]
    messages = history + [{"role": "user", "content": message}]

    response = anthropic.messages.create(model=Model1, max_tokens=1024, system=system_message, messages=messages, tools=anthropic_tools)

    while response.stop_reason == "tool_use":
        tool_result = handle_tool_calls_anthropic(response)
        messages.append({"role": "assistant", "content": response.content})
        messages.append(tool_result)
        response = anthropic.messages.create(model=Model1, max_tokens=1024, system=system_message, messages=messages, tools=anthropic_tools)

    with anthropic.messages.stream(model=Model1, max_tokens=1024, system=system_message, messages=messages) as s:
        result = ""
        for text in s.text_stream:
            result += text
            yield result

In [54]:
def handle_tool_calls_chat(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "look_up_python_docs":
            arguments = json.loads(tool_call.function.arguments)
            query = arguments.get('query')
            print(f"Tool called: look_up_python_docs('{query}')")  # add this
            result = look_up_python_docs(query)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

In [55]:
def handle_tool_calls_anthropic(response):
    tool_results = []
    for block in response.content:
        if block.type == "tool_use" and block.name == "look_up_python_docs":
            query = block.input.get('query')
            print(f"Tool called: look_up_python_docs('{query}')")  # add this
            result = look_up_python_docs(query)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result
            })
    return {"role": "user", "content": tool_results}

In [56]:
def stream_model(message, history, model):
    if model == "GPT":
        yield from chat(message, history)
    elif model == "Claude":
        yield from claude_chat(message, history)
    else:
        raise ValueError("Unknown model")

In [57]:
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")

view = gr.ChatInterface(
    fn=stream_model,
    title="Technical Q&A Assistant",
    additional_inputs=[model_selector],
    type="messages"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Tool called: look_up_python_docs('json.loads')
